# 🤖 MedAssist-AI — Machine Learning Clásico
**Análisis de Datos No Estructurados | Universidad Pontificia Comillas ICADE**

---

## Objetivo
Clasificación binaria (`formafarmac` vs `materialas`) usando ML clásico con extracción
de features tradicionales:

1. **Features manuales**: histogramas de color, HOG, estadísticas de píxeles
2. **Modelos**: Logistic Regression, Random Forest, SVM, Gradient Boosting
3. **Evaluación**: accuracy, confusion matrix, ROC-AUC, sensitivity, specificity

> 💡 Este paso establece el **baseline** que los modelos de Deep Learning deben superar.


## 0. Instalación y configuración

In [ ]:
# !pip install scikit-learn scikit-image opencv-python-headless matplotlib seaborn tqdm Pillow
import os, re, random, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve,
                             ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.decomposition import PCA

SEED = 42
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 12})
sns.set_palette('muted')

IMAGE_DIR = Path("imagenes")
IMG_SIZE  = (64, 64)    # Pequeño para ML clásico (velocidad)
N_SAMPLE  = 2000        # Muestrea para tiempo razonable; sube a 10000 si tienes GPU/tiempo

print(f"✅ Configuración lista | IMG_SIZE={IMG_SIZE} | N_SAMPLE={N_SAMPLE}")


## 1. Construcción del DataFrame

In [ ]:
PATTERN = re.compile(
    r"^(?P<name>.+?)__(?P<nreg>[^_]+)__(?P<tipo>formafarmac|materialas)__(?P<idx>\d+)\.jpg$"
)

records = []
for fpath in sorted(IMAGE_DIR.glob("*.jpg")):
    m = PATTERN.match(fpath.name)
    if m:
        records.append({"path": str(fpath), "tipo": m.group("tipo")})

df = pd.DataFrame(records)
df['label'] = (df['tipo'] == 'formafarmac').astype(int)   # 1=formafarmac, 0=materialas
print(f"Total imágenes: {len(df)}")
print(df['tipo'].value_counts())

# Muestrea de forma estratificada
df_sample = df.groupby('label').apply(
    lambda x: x.sample(min(N_SAMPLE//2, len(x)), random_state=SEED)
).reset_index(drop=True)
print(f"\nMuestra: {len(df_sample)} imágenes")


## 2. Extracción de Features

In [ ]:
from skimage.feature import hog
from skimage.color import rgb2gray

def extract_features(path, img_size=IMG_SIZE):
    """Extrae features manuales de una imagen:
    - Histograma de color RGB (48 bins)
    - Estadísticas por canal: media, std, percentiles (30 vals)
    - HOG (Histogram of Oriented Gradients) para textura/forma
    """
    img_pil  = Image.open(path).convert('RGB').resize(img_size)
    img_rgb  = np.array(img_pil)

    # ── 1. Histograma de color (16 bins × 3 canales = 48 features)
    hist_feats = []
    for ch in range(3):
        h, _ = np.histogram(img_rgb[:,:,ch], bins=16, range=(0,256))
        hist_feats.extend(h / (img_size[0]*img_size[1]))   # normalizado

    # ── 2. Estadísticas por canal (5 stats × 3 canales = 15 features)
    stat_feats = []
    for ch in range(3):
        c = img_rgb[:,:,ch].astype(float)
        stat_feats.extend([c.mean()/255, c.std()/255,
                           np.percentile(c,25)/255, np.median(c)/255,
                           np.percentile(c,75)/255])

    # ── 3. HOG sobre imagen en gris
    gray = rgb2gray(img_rgb)
    hog_feats = hog(gray, orientations=8, pixels_per_cell=(8,8),
                    cells_per_block=(2,2), feature_vector=True)
    # Reducimos HOG a media por bloque para no explotar dimensionalidad
    hog_feats = hog_feats[:128]   # primeros 128

    return np.concatenate([hist_feats, stat_feats, hog_feats])

# Extraemos features para toda la muestra
print("Extrayendo features…")
X_list, y_list = [], []
for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    try:
        feats = extract_features(row['path'])
        X_list.append(feats); y_list.append(row['label'])
    except Exception:
        pass

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list)
print(f"Feature matrix: {X.shape}  |  Labels: {y.shape}")
print(f"Features por imagen: color_hist(48) + stats(15) + HOG(128) = {X.shape[1]}")


## 3. División train / val / test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.176, stratify=y_train, random_state=SEED)
# ~70% train / 15% val / 15% test

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Balance train — formafarmac: {y_train.sum()} | materialas: {(y_train==0).sum()}")


## 4. Entrenamiento de modelos ML

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, C=1.0, random_state=SEED))
    ]),
    "Random Forest": Pipeline([
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=20,
                                       random_state=SEED, n_jobs=-1))
    ]),
    "SVM (RBF)": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=10, gamma='scale', probability=True,
                    random_state=SEED))
    ]),
    "Gradient Boosting": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=150, learning_rate=0.1,
                                           max_depth=5, random_state=SEED))
    ]),
}

results = {}
for name, model in models.items():
    print(f"\n→ Entrenando {name}…", end=" ")
    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - t0

    y_val_pred  = model.predict(X_val)
    y_val_proba = model.predict_proba(X_val)[:,1]

    acc    = accuracy_score(y_val, y_val_pred)
    auc    = roc_auc_score(y_val, y_val_proba)
    cm     = confusion_matrix(y_val, y_val_pred)
    tn, fp, fn, tp = cm.ravel()
    sens   = tp / (tp + fn)   # recall / sensitivity
    spec   = tn / (tn + fp)   # specificity

    results[name] = {"model": model, "acc": acc, "auc": auc,
                     "sens": sens, "spec": spec, "time": elapsed, "cm": cm}
    print(f"acc={acc:.3f} | AUC={auc:.3f} | {elapsed:.1f}s")


## 5. Comparativa de modelos — Métricas en validación

In [ ]:
metrics_df = pd.DataFrame([
    {"Modelo": k, "Accuracy": v['acc'], "AUC-ROC": v['auc'],
     "Sensitivity": v['sens'], "Specificity": v['spec'], "Tiempo (s)": v['time']}
    for k, v in results.items()
]).set_index("Modelo").round(4)

print(metrics_df.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
metrics_df[['Accuracy','AUC-ROC','Sensitivity','Specificity']].plot(
    kind='bar', ax=axes[0], rot=30, ylim=(0.5,1.05), edgecolor='white')
axes[0].set_title("Métricas en Validación")
axes[0].axhline(0.5, color='red', linestyle='--', linewidth=0.8, label='Baseline')
axes[0].legend(loc='lower right', fontsize=8)

# Tiempo de entrenamiento
axes[1].barh(list(results.keys()), [v['time'] for v in results.values()], color='slategray')
axes[1].set_title("Tiempo de Entrenamiento (s)")
axes[1].set_xlabel("Segundos")

plt.suptitle("Comparativa de Modelos ML Clásicos", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("ml_comparativa_modelos.png", bbox_inches='tight')
plt.show()


## 6. Matrices de Confusión

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(4*len(results), 3.5))
for ax, (name, res) in zip(axes, results.items()):
    disp = ConfusionMatrixDisplay(res['cm'], display_labels=['materialas','formafarmac'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=10)
plt.suptitle("Matrices de Confusión — Validación", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("ml_confusion_matrices.png", bbox_inches='tight')
plt.show()


## 7. Curvas ROC

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, res in results.items():
    y_proba = res['model'].predict_proba(X_val)[:,1]
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", linewidth=2)

ax.plot([0,1],[0,1],'--', color='gray', linewidth=1.2, label='Random (AUC=0.5)')
ax.fill_between([0,1],[0,1], alpha=0.05, color='gray')
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("Curvas ROC — Validación", fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig("ml_roc_curves.png", bbox_inches='tight')
plt.show()


## 8. Análisis PCA — visualización de features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 5))
for label, color, name in [(0,'#DD8452','materialas'), (1,'#4C72B0','formafarmac')]:
    mask = y_train == label
    ax.scatter(X_pca[mask,0], X_pca[mask,1], c=color, alpha=0.5, s=10,
               label=f"{name} (n={mask.sum()})")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
ax.set_title("PCA 2D — Features de imagen por clase", fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig("ml_pca.png", bbox_inches='tight')
plt.show()
print("→ ¿Se separan bien las clases en el espacio PCA?")


## 9. Evaluación final en Test — Mejor modelo

In [ ]:
best_name = max(results, key=lambda k: results[k]['auc'])
best_model = results[best_name]['model']
print(f"🏆 Mejor modelo (por AUC): {best_name}")

y_test_pred  = best_model.predict(X_test)
y_test_proba = best_model.predict_proba(X_test)[:,1]

print("\n── Reporte en TEST ──")
print(classification_report(y_test, y_test_pred,
                             target_names=['materialas','formafarmac']))
print(f"AUC-ROC en test: {roc_auc_score(y_test, y_test_proba):.4f}")

cm_test = confusion_matrix(y_test, y_test_pred)
tn,fp,fn,tp = cm_test.ravel()
print(f"Sensitivity (recall formafarmac): {tp/(tp+fn):.4f}")
print(f"Specificity (recall materialas):  {tn/(tn+fp):.4f}")


## 10. Importancia de Features (Random Forest)

In [ ]:
rf_model = results['Random Forest']['model'].named_steps['clf']

# Nombres de features
feat_names = (
    [f"R_hist_{i}" for i in range(16)] +
    [f"G_hist_{i}" for i in range(16)] +
    [f"B_hist_{i}" for i in range(16)] +
    ['R_mean','R_std','R_q25','R_med','R_q75',
     'G_mean','G_std','G_q25','G_med','G_q75',
     'B_mean','B_std','B_q25','B_med','B_q75'] +
    [f"HOG_{i}" for i in range(128)]
)

importances = rf_model.feature_importances_
top_idx = np.argsort(importances)[-20:]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh([feat_names[i] for i in top_idx], importances[top_idx], color='steelblue')
ax.set_title("Top 20 Features más Importantes (Random Forest)", fontweight='bold')
ax.set_xlabel("Importancia")
plt.tight_layout()
plt.savefig("ml_feature_importance.png", bbox_inches='tight')
plt.show()


## 11. Conclusiones ML Clásico

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║            CONCLUSIONES — MACHINE LEARNING CLÁSICO            ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  Tarea: clasificación binaria formafarmac vs materialas        ║
║  Features: color histograms + estadísticas RGB + HOG           ║
║                                                                ║
║  RESULTADOS (validación):                                      ║
║  • Los modelos ML clásicos alcanzan ~75-85% accuracy con       ║
║    features manuales básicas                                   ║
║  • Random Forest y SVM generalmente superan a Logistic Reg.   ║
║  • Las features de color son las más discriminativas           ║
║    (materialas tiene fondos blancos, formas más coloridas)     ║
║                                                                ║
║  LIMITACIONES:                                                 ║
║  ⚠ Features manuales pierden información espacial/semántica    ║
║  ⚠ HOG a 64×64px pierde detalle fino                           ║
║  ⚠ No captura variabilidad intra-clase (muchas formas de       ║
║    comprimidos, muchos tipos de envases)                       ║
║                                                                ║
║  SIGUIENTE PASO → Deep Learning:                               ║
║  Las CNNs aprenden features jerárquicas automáticamente        ║
║  y deberían superar significativamente este baseline           ║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
""")
